# SDCP Ablation Study

Tests 3 component ablations on **TruthfulQA** and **MMLU**:

| Variant | What's removed | How |
|---------|---------------|-----|
| `SDCP-full` | nothing | baseline |
| `w/o P_neg` | negative prior | set γ=0 in scoring; remove P_neg from prompt |
| `w/o cert` | certainty routing | always use the *uncertain* path |
| `w/o P_pos` | positive prior | no P_pos in scoring or prompt; standard cosine retrieval |

**Estimated runtime:** ~3–4 hours total (4-bit, TQA=615Q + MMLU=1596Q × 3 variants)

In [1]:
# ── Cell 1: Imports & Config ──────────────────────────────────────────────────
import os, sys, gc, json, re, torch, faiss, numpy as np
from datetime import datetime
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_from_disk
import pandas as pd

BASE_DIR     = '/home/user/RAG_Best_Practices/RAG_best_practices-main'
MODELS_DIR   = '/home/user/RAG_Best_Practices/models'
DATASETS_DIR = '/home/user/RAG_Best_Practices/datasets'
OUTPUT_DIR   = '/home/user/RAG_Best_Practices/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED   = 42
CHOICE_LABELS = ['A', 'B', 'C', 'D']
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Full SDCP params (paper values)
SDCP_PARAMS = dict(
    cert_threshold=0.65,
    alpha=0.45, beta=0.35, gamma=0.20,
    top_k_retrieve=15, top_k_context=4,
    max_gen_tokens=25, max_pos_tokens=20, max_neg_tokens=20,
)
N_TRUTHFULQA     = 615
MMLU_PER_SUBJECT = 28

print('Config OK')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

Config OK
GPU: Quadro RTX 8000


In [2]:
# ── Cell 2: Load Datasets ─────────────────────────────────────────────────────
print('Loading TruthfulQA...')
tqa_raw = load_from_disk(f'{DATASETS_DIR}/truthfulqa').to_pandas()
tqa_all = tqa_raw[['question','best_answer','correct_answers','incorrect_answers']].copy()
tqa_all['correct_answers']   = tqa_all['correct_answers'].apply(lambda x: x.tolist() if hasattr(x,'tolist') else [x])
tqa_all['incorrect_answers'] = tqa_all['incorrect_answers'].apply(lambda x: x.tolist() if hasattr(x,'tolist') else [x])
tqa_all['best_answer']       = tqa_all['best_answer'].apply(lambda x: [x] if x else [])
tqa_all = tqa_all[(tqa_all['correct_answers'].apply(len)>1) &
                  (tqa_all['incorrect_answers'].apply(len)>1)].reset_index(drop=True)
tqa_idx = tqa_all.sample(n=N_TRUTHFULQA, random_state=RANDOM_SEED).index
tqa     = tqa_all.loc[tqa_idx].reset_index(drop=True)
tqa_kb  = tqa_all.drop(tqa_idx).reset_index(drop=True)
del tqa_raw; gc.collect()
print(f'  TQA test={len(tqa)} KB={len(tqa_kb)}')

print('Loading MMLU...')
mmlu_raw = load_from_disk(f'{DATASETS_DIR}/mmlu').to_pandas()
def mmlu_to_unified(row):
    choices = list(row['choices']); ans_idx = int(row['answer'])
    correct = choices[ans_idx]
    incorrect = [choices[i] for i in range(len(choices)) if i != ans_idx]
    fq = row['question']+'\n'+'\n'.join(f'{CHOICE_LABELS[i]}) {choices[i]}' for i in range(len(choices)))
    return pd.Series({'question':fq,'question_plain':row['question'],'subject':row['subject'],
                      'best_answer':[correct],'correct_answers':[correct],
                      'incorrect_answers':incorrect,'answer_idx':ans_idx,'choices':choices})
mmlu_test_parts, mmlu_kb_parts = [], []
for subject, group in mmlu_raw.groupby('subject'):
    group = group.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n_test = min(MMLU_PER_SUBJECT, len(group))
    mmlu_test_parts.append(group.iloc[:n_test])
    if len(group) > n_test: mmlu_kb_parts.append(group.iloc[n_test:])
mmlu    = pd.concat(mmlu_test_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
mmlu_kb = pd.concat(mmlu_kb_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
del mmlu_raw; gc.collect()
print(f'  MMLU test={len(mmlu)} KB={len(mmlu_kb)}')

Loading TruthfulQA...
  TQA test=615 KB=100
Loading MMLU...
  MMLU test=1596 KB=228


In [3]:
# ── Cell 3: Load Model ────────────────────────────────────────────────────────
gc.collect(); torch.cuda.empty_cache()
print('Loading Mistral-7B (4-bit)...')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm = AutoModelForCausalLM.from_pretrained(
    f'{MODELS_DIR}/mistral-7b', quantization_config=bnb, device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(f'{MODELS_DIR}/mistral-7b', padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
print('Loading MiniLM...')
embed_model = SentenceTransformer(f'{MODELS_DIR}/minilm').to(DEVICE)
print('Models ready!')

Loading Mistral-7B (4-bit)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading MiniLM...
Models ready!


In [4]:
# ── Cell 4: Utilities (same as Full_Dataset_4bit) ─────────────────────────────
INST_S = '[INST]'
INST_E = '[/INST]'
SYS    = 'You are a truthful expert question-answering bot and should correctly and concisely answer the following question'

def generate(prompts, max_new_tokens=25, num_beams=2):
    enc = tokenizer(prompts, return_tensors='pt', padding=True,
                    truncation=True, max_length=2048).to(DEVICE)
    input_len = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = llm.generate(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
                           max_new_tokens=max_new_tokens, num_beams=num_beams,
                           pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(r[input_len:], skip_special_tokens=True).strip() or 'I have no comment' for r in out]

def get_token_probs(prompt, max_new_tokens=20):
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                           return_dict_in_generate=True, output_scores=True,
                           pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out.sequences[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    cert = 0.0
    if out.scores:
        probs = torch.softmax(out.scores[0][0], dim=-1)
        top2  = torch.topk(probs, 2).values
        cert  = (top2[0] - top2[1]).item()
    return text, cert

def clean_response(resp):
    for stop in ['\nQuestion:','\nQ:','\n---','\n\n','\nContext:','\nFor the question',
                 '\nCommon','\nMy initial','\nVerified','\nRefined']:
        if stop in resp: resp = resp[:resp.index(stop)]
    return resp.strip().strip('"').strip("'") or 'I have no comment'

def build_index(dataset):
    embs = embed_model.encode(dataset['question'].tolist(), show_progress_bar=True, batch_size=64)
    embs = np.array(embs, dtype=np.float32); faiss.normalize_L2(embs)
    idx  = faiss.IndexFlatIP(embs.shape[1]); idx.add(embs)
    return idx

def cosine_retrieve(query, faiss_idx, kb, k=15):
    q_emb = np.array(embed_model.encode([query], show_progress_bar=False), dtype=np.float32)
    faiss.normalize_L2(q_emb)
    _, idxs = faiss_idx.search(q_emb, k)
    return [kb.iloc[i] for i in idxs[0] if i < len(kb)]

def compute_metrics(generated, references):
    scorer = rs_module.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1s, r2s, rls, ecss = [], [], [], []
    for gen, refs in zip(generated, references):
        best = [0,0,0]
        for ref in refs:
            if not ref: continue
            s = scorer.score(ref, gen)
            best[0]=max(best[0],s['rouge1'].fmeasure)
            best[1]=max(best[1],s['rouge2'].fmeasure)
            best[2]=max(best[2],s['rougeL'].fmeasure)
        r1s.append(best[0]*100); r2s.append(best[1]*100); rls.append(best[2]*100)
        try:
            embs = embed_model.encode([refs[0], gen])
            ecss.append(float(cosine_similarity([embs[0]],[embs[1]])[0][0])*100)
        except: ecss.append(0.0)
    return np.array(r1s), np.array(r2s), np.array(rls), np.array(ecss)

def pack_result(variant_name, dataset_name, generated, references):
    r1, r2, rl, ecs = compute_metrics(generated, references)
    res = {'variant': variant_name, 'dataset': dataset_name,
           'R1': round(float(r1.mean()),3), 'R2': round(float(r2.mean()),3),
           'RL': round(float(rl.mean()),3), 'ECS': round(float(ecs.mean()),3)}
    print(f'  [{variant_name}] R1={res["R1"]} R2={res["R2"]} RL={res["RL"]} ECS={res["ECS"]}')
    return res

print('Utilities ready ✓')

Utilities ready ✓


In [5]:
# ── Cell 5: SDCP Variants ─────────────────────────────────────────────────────
# Single function that runs all 4 variants based on flags.

def run_sdcp_variant(test_data, kb_data, dataset_name, variant='full',
                     use_pneg=True, use_cert=True, use_ppos=True, params=None):
    """
    variant:   label for output
    use_pneg:  if False → γ=0, no P_neg in prompt (w/o P_neg ablation)
    use_cert:  if False → always use uncertain path (w/o cert ablation)
    use_ppos:  if False → no P_pos in scoring, standard cosine only (w/o P_pos ablation)
    """
    if params is None: params = SDCP_PARAMS
    p = params.copy()
    if not use_pneg: p['gamma'] = 0.0   # zero out negative prior weight

    print(f'\n=== {variant} | {dataset_name} | test={len(test_data)} KB={len(kb_data)} ===')
    print(f'    use_ppos={use_ppos} use_pneg={use_pneg} use_cert={use_cert} γ={p["gamma"]}')

    faiss_idx = build_index(kb_data)
    generated, references = [], []

    for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc=variant):
        query       = row['question']
        best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

        # ── Step 1: self-distillation (conditional)
        if use_ppos:
            pos_prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer concisely:{INST_E}'
            p_pos, cert = get_token_probs(pos_prompt, max_new_tokens=p['max_pos_tokens'])
            p_pos = clean_response(p_pos)
        else:
            p_pos, cert = None, 0.5  # no prior → cert irrelevant

        if use_pneg and use_ppos:
            q_plain = query.split('\n')[0]
            if dataset_name in ['MMLU', 'ARC']:
                neg_prompt = (f'{INST_S}What is a plausible but INCORRECT answer students '
                              f'commonly give?\nQuestion: {q_plain}\nWrong answer (very short):{INST_E}')
            else:
                neg_prompt = (f'{INST_S}What is a common misconception about this topic?\n'
                              f'Question: {q_plain}\nWrong belief (very short):{INST_E}')
            p_neg = clean_response(generate([neg_prompt], max_new_tokens=p['max_neg_tokens'], num_beams=1)[0])
        else:
            p_neg = None

        # ── Step 2: retrieval (contrastive or cosine-only)
        retrieved = cosine_retrieve(query, faiss_idx, kb_data, k=p['top_k_retrieve'])
        prompt = None

        if retrieved:
            sentences = []
            for doc in retrieved:
                sentences.append(doc['question'])
                ba = doc['best_answer']
                ba_t = ba[0] if isinstance(ba, list) and ba else str(ba)
                if ba_t and len(ba_t) > 3: sentences.append(ba_t)
            sentences = [s for s in sentences if s and len(s.strip()) > 5]

            if sentences:
                s_embs = embed_model.encode(sentences, show_progress_bar=False)
                q_emb  = embed_model.encode([query], show_progress_bar=False)
                q_sims = cosine_similarity(s_embs, q_emb).flatten()

                if use_ppos and p_pos:
                    pos_emb = embed_model.encode([p_pos], show_progress_bar=False)
                    p_sims  = cosine_similarity(s_embs, pos_emb).flatten()
                else:
                    p_sims  = np.zeros(len(sentences))  # no P_pos → β term = 0

                if use_pneg and p_neg:
                    neg_emb = embed_model.encode([p_neg], show_progress_bar=False)
                    n_sims  = cosine_similarity(s_embs, neg_emb).flatten()
                else:
                    n_sims  = np.zeros(len(sentences))  # γ=0

                scores  = p['alpha']*q_sims + p['beta']*p_sims - p['gamma']*n_sims
                context = ' '.join([sentences[i] for i in np.argsort(-scores)[:p['top_k_context']]])

                kb_ex       = retrieved[0]
                kb_correct  = kb_ex['best_answer'][0] if isinstance(kb_ex['best_answer'],list) and kb_ex['best_answer'] else str(kb_ex['best_answer'])
                kb_incorrect= kb_ex['incorrect_answers'][0] if isinstance(kb_ex['incorrect_answers'],list) and kb_ex['incorrect_answers'] else ''
                kb_q        = kb_ex.get('question_plain', kb_ex['question'])

                # ── Step 3: adaptive prompt
                # w/o cert → always uncertain path
                # w/o P_pos → no prior in prompt, use basic context prompt
                if not use_ppos:
                    prompt = (f'{INST_S}{SYS}\nContext: {context}\n'
                              f'Question: {query}\nAnswer:{INST_E}')
                elif not use_cert or cert < p['cert_threshold']:
                    pneg_line = f'\nCommon mistake to avoid: {p_neg}' if p_neg else ''
                    prompt = (f'{INST_S}{SYS}\nRetrieved context: {context}\n'
                              f'Example — Q: {kb_q}\n  Correct: {kb_correct}\n  Incorrect: {kb_incorrect}\n'
                              f'My initial thought: {p_pos}{pneg_line}\n'
                              f'Question: {query}\nVerified answer:{INST_E}')
                else:
                    pneg_line = f'\nCommon mistake to avoid: {p_neg}' if p_neg else ''
                    prompt = (f'{INST_S}{SYS}\nContext: {context}\n'
                              f'Initial answer: {p_pos}{pneg_line}\n'
                              f'Question: {query}\nRefined answer:{INST_E}')

        if prompt is None:
            prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

        resp = generate([prompt], max_new_tokens=p['max_gen_tokens'], num_beams=2)
        generated.append(clean_response(resp[0]))
        references.append(best_answer)
        if idx % 50 == 0: gc.collect(); torch.cuda.empty_cache()

    return pack_result(variant, dataset_name, generated, references)

print('SDCP variant function ready ✓')

SDCP variant function ready ✓


In [6]:
# ── Cell 6: RUN — TruthfulQA Ablation ────────────────────────────────────────
import time
ablation_results = {}

print('='*60)
print(f'TruthfulQA Ablation (test={len(tqa)}, KB={len(tqa_kb)})')
print('='*60)

# Full SDCP (reference — should match full_4bit numbers: R1=34.26)
t0 = time.time()
ablation_results['full_TQA'] = run_sdcp_variant(
    tqa, tqa_kb, 'TQA', variant='SDCP-full',
    use_ppos=True, use_pneg=True, use_cert=True
)
print(f'  Done in {(time.time()-t0)/60:.1f} min')

# w/o P_neg (γ=0)
t0 = time.time()
ablation_results['no_pneg_TQA'] = run_sdcp_variant(
    tqa, tqa_kb, 'TQA', variant='w/o P_neg (γ=0)',
    use_ppos=True, use_pneg=False, use_cert=True
)
print(f'  Done in {(time.time()-t0)/60:.1f} min')

# w/o cert (always uncertain path)
t0 = time.time()
ablation_results['no_cert_TQA'] = run_sdcp_variant(
    tqa, tqa_kb, 'TQA', variant='w/o cert (always uncertain)',
    use_ppos=True, use_pneg=True, use_cert=False
)
print(f'  Done in {(time.time()-t0)/60:.1f} min')

# w/o P_pos (no prior — standard cosine only)
t0 = time.time()
ablation_results['no_ppos_TQA'] = run_sdcp_variant(
    tqa, tqa_kb, 'TQA', variant='w/o P_pos (no prior)',
    use_ppos=False, use_pneg=False, use_cert=False
)
print(f'  Done in {(time.time()-t0)/60:.1f} min')

print('\nTQA ablation done!')

TruthfulQA Ablation (test=615, KB=100)

=== SDCP-full | TQA | test=615 KB=100 ===
    use_ppos=True use_pneg=True use_cert=True γ=0.2


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

SDCP-full: 100%|██████████| 615/615 [41:02<00:00,  4.00s/it]


  [SDCP-full] R1=34.598 R2=18.647 RL=30.938 ECS=65.425
  Done in 41.1 min

=== w/o P_neg (γ=0) | TQA | test=615 KB=100 ===
    use_ppos=True use_pneg=False use_cert=True γ=0.0


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

w/o cert (always uncertain): 100%|██████████| 615/615 [41:13<00:00,  4.02s/it]


  [w/o cert (always uncertain)] R1=36.305 R2=20.495 RL=32.782 ECS=65.65
  Done in 41.3 min

=== w/o P_pos (no prior) | TQA | test=615 KB=100 ===
    use_ppos=False use_pneg=False use_cert=False γ=0.0


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

w/o P_pos (no prior): 100%|██████████| 615/615 [19:44<00:00,  1.93s/it]


  [w/o P_pos (no prior)] R1=30.302 R2=15.776 RL=26.965 ECS=60.252
  Done in 19.8 min

TQA ablation done!


In [7]:
# ── Cell 7: RUN — MMLU Ablation ──────────────────────────────────────────────
print('='*60)
print(f'MMLU Ablation (test={len(mmlu)}, KB={len(mmlu_kb)})')
print('='*60)

t0 = time.time()
ablation_results['full_MMLU'] = run_sdcp_variant(
    mmlu, mmlu_kb, 'MMLU', variant='SDCP-full',
    use_ppos=True, use_pneg=True, use_cert=True
)
print(f'  Done in {(time.time()-t0)/60:.1f} min')

t0 = time.time()
ablation_results['no_pneg_MMLU'] = run_sdcp_variant(
    mmlu, mmlu_kb, 'MMLU', variant='w/o P_neg (γ=0)',
    use_ppos=True, use_pneg=False, use_cert=True
)
print(f'  Done in {(time.time()-t0)/60:.1f} min')

t0 = time.time()
ablation_results['no_cert_MMLU'] = run_sdcp_variant(
    mmlu, mmlu_kb, 'MMLU', variant='w/o cert (always uncertain)',
    use_ppos=True, use_pneg=True, use_cert=False
)
print(f'  Done in {(time.time()-t0)/60:.1f} min')

t0 = time.time()
ablation_results['no_ppos_MMLU'] = run_sdcp_variant(
    mmlu, mmlu_kb, 'MMLU', variant='w/o P_pos (no prior)',
    use_ppos=False, use_pneg=False, use_cert=False
)
print(f'  Done in {(time.time()-t0)/60:.1f} min')

print('\nMMELU ablation done!')

MMLU Ablation (test=1596, KB=228)

=== SDCP-full | MMLU | test=1596 KB=228 ===
    use_ppos=True use_pneg=True use_cert=True γ=0.2


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

SDCP-full: 100%|██████████| 1596/1596 [1:52:01<00:00,  4.21s/it]


  [SDCP-full] R1=32.539 R2=23.807 RL=31.649 ECS=47.884
  Done in 112.2 min

=== w/o P_neg (γ=0) | MMLU | test=1596 KB=228 ===
    use_ppos=True use_pneg=False use_cert=True γ=0.0


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

...

w/o cert (always uncertain): 100%|██████████| 1596/1596 [1:54:23<00:00,  4.30s/it] 


  [w/o cert (always uncertain)] R1=35.693 R2=27.129 RL=34.875 ECS=48.903
  Done in 114.6 min

=== w/o P_pos (no prior) | MMLU | test=1596 KB=228 ===
    use_ppos=False use_pneg=False use_cert=False γ=0.0


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

w/o P_pos (no prior): 100%|██████████| 1596/1596 [55:59<00:00,  2.11s/it]


  [w/o P_pos (no prior)] R1=36.853 R2=27.423 RL=35.877 ECS=49.859
  Done in 56.2 min

MMELU ablation done!


In [8]:
# ── Cell 8: Results Table & Save ─────────────────────────────────────────────
variants = [
    ('SDCP-full',               'full_TQA',    'full_MMLU'),
    ('w/o P_neg (γ=0)',         'no_pneg_TQA', 'no_pneg_MMLU'),
    ('w/o cert (always uncert)','no_cert_TQA', 'no_cert_MMLU'),
    ('w/o P_pos (no prior)',    'no_ppos_TQA', 'no_ppos_MMLU'),
]

print('\n' + '='*65)
print('ABLATION RESULTS')
print('='*65)
print(f'{"Variant":<30} {"TQA R1":>8} {"TQA R2":>8} {"MMLU R1":>9} {"MMLU R2":>9}')
print('-'*65)
for label, tqa_key, mmlu_key in variants:
    t = ablation_results.get(tqa_key, {})
    m = ablation_results.get(mmlu_key, {})
    print(f'{label:<30} {t.get("R1",0):>8.2f} {t.get("R2",0):>8.2f} '
          f'{m.get("R1",0):>9.2f} {m.get("R2",0):>9.2f}')

# Save
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = f'{OUTPUT_DIR}/ablation_results_{ts}.json'
with open(out_path, 'w') as f:
    json.dump(ablation_results, f, indent=2)
print(f'\nSaved → {out_path}')
print('\n*** Copy these numbers to the ablation table in emnlp2023.tex ***')


ABLATION RESULTS
Variant                          TQA R1   TQA R2   MMLU R1   MMLU R2
-----------------------------------------------------------------
SDCP-full                         34.60    18.65     32.54     23.81
w/o P_neg (γ=0)                   33.65    17.40     32.63     23.80
w/o cert (always uncert)          36.30    20.50     35.69     27.13
w/o P_pos (no prior)              30.30    15.78     36.85     27.42

Saved → /home/user/RAG_Best_Practices/outputs/ablation_results_20260430_200833.json

*** Copy these numbers to the ablation table in emnlp2023.tex ***
